# Chapter 3 Lab Solutions: Multiagent Search (Real-World Assessment)

This notebook provides complete reference implementations for the real-world Chapter 3 lab scenarios.

```{admonition} Instructor Note
:class: warning
This file is intentionally not referenced in the website table of contents.
```

In [ ]:
import math
import random
import time
from dataclasses import dataclass
from copy import deepcopy

random.seed(7)
print('Solution notebook environment ready')

## Exercise 1 Solution: Emergency Route Planning with Minimax

MAX chooses a route and MIN chooses worst-case traffic disruption.

In [ ]:
class EmergencyRouteState:
    def __init__(self, stage='dispatch', route=None, traffic_event=None):
        self.stage = stage
        self.route = route
        self.traffic_event = traffic_event

    def available_actions(self):
        if self.stage == 'dispatch':
            return ['highway', 'arterial', 'local']
        if self.stage == 'traffic':
            return ['none', 'incident', 'signal_failure']
        return []

    def apply(self, action):
        if self.stage == 'dispatch':
            return EmergencyRouteState(stage='traffic', route=action)
        if self.stage == 'traffic':
            return EmergencyRouteState(stage='terminal', route=self.route, traffic_event=action)
        return self

    def is_terminal(self):
        return self.stage == 'terminal'

    def utility(self):
        base = {'highway': 80, 'arterial': 70, 'local': 60}[self.route]
        penalty = {'none': 0, 'incident': 25, 'signal_failure': 15}[self.traffic_event]
        return base - penalty


def minimax_route(state, maximizing=True):
    if state.is_terminal():
        return state.utility(), None

    actions = state.available_actions()
    if maximizing:
        best_val = float('-inf')
        best_action = None
        for a in actions:
            v, _ = minimax_route(state.apply(a), maximizing=False)
            if v > best_val:
                best_val = v
                best_action = a
        return best_val, best_action
    else:
        best_val = float('inf')
        best_action = None
        for a in actions:
            v, _ = minimax_route(state.apply(a), maximizing=True)
            if v < best_val:
                best_val = v
                best_action = a
        return best_val, best_action


root = EmergencyRouteState()
value, action = minimax_route(root, maximizing=True)
print(f'Best emergency route: {action}, guaranteed score: {value}')

## Exercise 2 Solution: Cybersecurity Defense with Alpha-Beta

Compare plain minimax and alpha-beta for defender-attacker planning.

In [ ]:
class CyberDefenseState:
    def __init__(self, stage='defense', defense=None, attack=None):
        self.stage = stage
        self.defense = defense
        self.attack = attack

    def available_actions(self):
        if self.stage == 'defense':
            return ['patch_critical', 'harden_network', 'increase_monitoring']
        if self.stage == 'attack':
            return ['phishing_chain', 'credential_stuffing', 'lateral_movement']
        return []

    def apply(self, action):
        if self.stage == 'defense':
            return CyberDefenseState(stage='attack', defense=action)
        if self.stage == 'attack':
            return CyberDefenseState(stage='terminal', defense=self.defense, attack=action)
        return self

    def is_terminal(self):
        return self.stage == 'terminal'

    def utility(self):
        payoff = {
            ('patch_critical', 'phishing_chain'): 60,
            ('patch_critical', 'credential_stuffing'): 75,
            ('patch_critical', 'lateral_movement'): 45,
            ('harden_network', 'phishing_chain'): 70,
            ('harden_network', 'credential_stuffing'): 55,
            ('harden_network', 'lateral_movement'): 80,
            ('increase_monitoring', 'phishing_chain'): 85,
            ('increase_monitoring', 'credential_stuffing'): 65,
            ('increase_monitoring', 'lateral_movement'): 60,
        }
        return payoff[(self.defense, self.attack)]


def minimax_cyber(state, maximizing=True, stats=None):
    if stats is None:
        stats = {'nodes': 0}
    stats['nodes'] += 1

    if state.is_terminal():
        return state.utility(), None, stats

    actions = state.available_actions()
    if maximizing:
        best_v, best_a = float('-inf'), None
        for a in actions:
            v, _, _ = minimax_cyber(state.apply(a), False, stats)
            if v > best_v:
                best_v, best_a = v, a
        return best_v, best_a, stats
    else:
        best_v, best_a = float('inf'), None
        for a in actions:
            v, _, _ = minimax_cyber(state.apply(a), True, stats)
            if v < best_v:
                best_v, best_a = v, a
        return best_v, best_a, stats


def alpha_beta_cyber(state, alpha=float('-inf'), beta=float('inf'), maximizing=True, stats=None):
    if stats is None:
        stats = {'nodes': 0, 'prunes': 0}
    stats['nodes'] += 1

    if state.is_terminal():
        return state.utility(), None, stats

    actions = state.available_actions()
    if maximizing:
        best_v, best_a = float('-inf'), None
        for a in actions:
            v, _, _ = alpha_beta_cyber(state.apply(a), alpha, beta, False, stats)
            if v > best_v:
                best_v, best_a = v, a
            alpha = max(alpha, best_v)
            if beta <= alpha:
                stats['prunes'] += 1
                break
        return best_v, best_a, stats
    else:
        best_v, best_a = float('inf'), None
        for a in actions:
            v, _, _ = alpha_beta_cyber(state.apply(a), alpha, beta, True, stats)
            if v < best_v:
                best_v, best_a = v, a
            beta = min(beta, best_v)
            if beta <= alpha:
                stats['prunes'] += 1
                break
        return best_v, best_a, stats


state = CyberDefenseState()
t0 = time.time()
v1, a1, s1 = minimax_cyber(state)
t1 = time.time() - t0
t0 = time.time()
v2, a2, s2 = alpha_beta_cyber(state)
t2 = time.time() - t0

print('Minimax ->', a1, v1, s1, f'time={t1:.6f}s')
print('Alpha-Beta ->', a2, v2, s2, f'time={t2:.6f}s')

## Exercise 3 Solution: Warehouse Dispatch (Depth-Limited Search)

Use a depth-limited alpha-beta with a domain-specific evaluation function.

In [ ]:
class WarehouseDispatchState:
    def __init__(self, backlog=12, battery=100, congestion=0, depth=0, max_depth=4):
        self.backlog = backlog
        self.battery = battery
        self.congestion = congestion
        self.depth = depth
        self.max_depth = max_depth

    def available_actions(self):
        return ['shortest_path', 'energy_saving_route', 'batch_pick_strategy']

    def disturbance_actions(self):
        return ['none', 'aisle_block', 'battery_drop']

    def apply(self, action):
        if action == 'shortest_path':
            return WarehouseDispatchState(max(0, self.backlog - 4), max(0, self.battery - 18), min(10, self.congestion + 2), self.depth + 1, self.max_depth)
        if action == 'energy_saving_route':
            return WarehouseDispatchState(max(0, self.backlog - 2), max(0, self.battery - 8), max(0, self.congestion - 1), self.depth + 1, self.max_depth)
        return WarehouseDispatchState(max(0, self.backlog - 3), max(0, self.battery - 12), min(10, self.congestion + 1), self.depth + 1, self.max_depth)

    def apply_disturbance(self, event):
        if event == 'none':
            return self
        if event == 'aisle_block':
            return WarehouseDispatchState(self.backlog + 1, self.battery, min(10, self.congestion + 2), self.depth, self.max_depth)
        return WarehouseDispatchState(self.backlog, max(0, self.battery - 7), self.congestion, self.depth, self.max_depth)

    def is_terminal(self):
        return self.backlog == 0 or self.battery <= 5 or self.depth >= self.max_depth


def evaluate_dispatch(state):
    # Higher is better
    return 100 - (6 * state.backlog) + (0.8 * state.battery) - (4 * state.congestion)


def alpha_beta_depth_limited_dispatch(state, alpha, beta, depth, max_depth, maximizing):
    if state.is_terminal() or depth >= max_depth:
        return evaluate_dispatch(state), None

    if maximizing:
        best_val, best_act = float('-inf'), None
        for act in state.available_actions():
            s2 = state.apply(act)
            val, _ = alpha_beta_depth_limited_dispatch(s2, alpha, beta, depth + 1, max_depth, False)
            if val > best_val:
                best_val, best_act = val, act
            alpha = max(alpha, best_val)
            if beta <= alpha:
                break
        return best_val, best_act
    else:
        best_val, best_evt = float('inf'), None
        for evt in state.disturbance_actions():
            s2 = state.apply_disturbance(evt)
            val, _ = alpha_beta_depth_limited_dispatch(s2, alpha, beta, depth + 1, max_depth, True)
            if val < best_val:
                best_val, best_evt = val, evt
            beta = min(beta, best_val)
            if beta <= alpha:
                break
        return best_val, best_evt


dispatch = WarehouseDispatchState()
v, a = alpha_beta_depth_limited_dispatch(dispatch, float('-inf'), float('inf'), 0, 4, True)
print(f'Best dispatch action: {a}, estimated value: {v:.2f}')

## Exercise 4 Solution: Dynamic Pricing with MCTS

Complete MCTS with UCB1 and random rollout policy.

In [ ]:
class PricingMarketState:
    def __init__(self, demand_level=0.6, competitor_pressure=0.5, step=0, max_steps=6):
        self.demand_level = demand_level
        self.competitor_pressure = competitor_pressure
        self.step = step
        self.max_steps = max_steps

    def available_moves(self):
        return ['price_up', 'price_down', 'hold']

    def apply(self, action):
        demand = self.demand_level
        pressure = self.competitor_pressure
        if action == 'price_up':
            demand = max(0.0, demand - 0.1)
        elif action == 'price_down':
            demand = min(1.0, demand + 0.1)
        return PricingMarketState(demand, pressure, self.step + 1, self.max_steps)

    def random_market_update(self):
        delta = random.choice([-0.05, 0.0, 0.05])
        pressure = min(1.0, max(0.0, self.competitor_pressure + random.choice([-0.05, 0.0, 0.05])))
        demand = min(1.0, max(0.0, self.demand_level + delta))
        return PricingMarketState(demand, pressure, self.step, self.max_steps)

    def is_terminal(self):
        return self.step >= self.max_steps

    def reward(self):
        margin = 0.5 + (0.2 if self.demand_level < 0.5 else -0.1)
        revenue_signal = self.demand_level * (1.0 - self.competitor_pressure)
        return 100 * (margin + revenue_signal)


class MCTSNode:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.wins = 0.0
        self.visits = 0
        self.untried_actions = state.available_moves()[:]

    def ucb1(self, c=1.41):
        if self.visits == 0:
            return float('inf')
        parent_visits = max(1, self.parent.visits if self.parent else 1)
        exploit = self.wins / self.visits
        explore = c * math.sqrt(math.log(parent_visits) / self.visits)
        return exploit + explore

    def select_child(self):
        return max(self.children, key=lambda child: child.ucb1())

    def add_child(self, action, state):
        child = MCTSNode(state, parent=self, action=action)
        self.untried_actions.remove(action)
        self.children.append(child)
        return child

    def update(self, result):
        self.visits += 1
        self.wins += result


def simulate_random_market(state):
    s = deepcopy(state)
    total = 0.0
    while not s.is_terminal():
        a = random.choice(s.available_moves())
        s = s.apply(a).random_market_update()
        total += s.reward()
    return total


def mcts_search(root_state, num_simulations=500):
    root = MCTSNode(root_state)

    for _ in range(num_simulations):
        node = root

        # Selection
        while not node.untried_actions and node.children:
            node = node.select_child()

        # Expansion
        if node.untried_actions:
            action = random.choice(node.untried_actions)
            next_state = node.state.apply(action).random_market_update()
            node = node.add_child(action, next_state)

        # Simulation
        result = simulate_random_market(node.state)

        # Backpropagation
        while node is not None:
            node.update(result)
            node = node.parent

    best = max(root.children, key=lambda c: c.visits) if root.children else None
    return best.action if best else None


market = PricingMarketState()
best_price_action = mcts_search(market, num_simulations=600)
print('Best pricing action from MCTS:', best_price_action)

## Exercise 5 Solution: Cross-Domain Tournament

Run a small round-robin tournament across strategies in one scenario factory.

In [ ]:
def play_scenario(strategy_a, strategy_b, scenario_factory):
    state = scenario_factory()
    turn = 0
    while hasattr(state, 'is_terminal') and not state.is_terminal() and turn < 20:
        actions = state.available_actions() if hasattr(state, 'available_actions') else state.available_moves()
        if not actions:
            break
        action = strategy_a(state) if turn % 2 == 0 else strategy_b(state)
        if action not in actions:
            action = random.choice(actions)
        state = state.apply(action)
        turn += 1
    return state


def tournament(strategies, scenario_factory, num_games=20):
    names = list(strategies.keys())
    scores = {name: 0.0 for name in names}

    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i == j:
                continue
            for _ in range(num_games):
                end_state = play_scenario(strategies[a], strategies[b], scenario_factory)
                if hasattr(end_state, 'utility') and hasattr(end_state, 'stage') and end_state.stage == 'terminal':
                    scores[a] += end_state.utility()
                elif hasattr(end_state, 'reward'):
                    scores[a] += end_state.reward()
                else:
                    scores[a] += 0.0
    return scores


strategy_bank = {
    'Random': lambda s: random.choice(s.available_actions()),
    'HeuristicRoute': lambda s: 'highway' if 'highway' in s.available_actions() else random.choice(s.available_actions()),
}

scores = tournament(strategy_bank, EmergencyRouteState, num_games=15)
print('Tournament scores:', scores)

## Challenge Solution: Iterative Deepening for Real-Time Decisions

In [ ]:
def iterative_deepening_decision(initial_state, decision_fn, time_limit=2.0):
    start = time.time()
    depth = 1
    best = (None, None, 0)

    while time.time() - start < time_limit:
        value, action = decision_fn(initial_state, depth)
        best = (value, action, depth)
        depth += 1

    return best


def depth_limited_route_decision(state, depth):
    # Depth ignored for this tiny game tree, kept for API consistency
    return minimax_route(state, maximizing=True)


best_value, best_action, reached_depth = iterative_deepening_decision(
    EmergencyRouteState(), depth_limited_route_decision, time_limit=0.05
)
print(f'Iterative deepening best action: {best_action}, value: {best_value}, depth reached: {reached_depth}')

## Wrap-Up Notes

These solutions are references, not the only valid answers. Students should still discuss modeling assumptions, data limitations, and algorithm-selection rationale for each scenario.

Suggested extension: run sensitivity analyses by perturbing reward tables and disturbance rates.